In [0]:
# =============================================================
# NOTEBOOK: 00_environment_validation
# PURPOSE:  Validate all connections and environment setup
# AUTHOR:   Your Name
# DATE:     Today's date
# PROJECT:  RetailMax Lakehouse
# =============================================================

print("=" * 60)
print("RetailMax Lakehouse - Environment Validation")
print("=" * 60)
print(f"Spark Version: {spark.version}")
print("=" * 60)

In [0]:
# ── CELL 2: Retrieve Secrets from Key Vault ──────────────────
# This tests if our Secret Scope (Day 1) is working correctly

scope_name = "kv-retailmax"

client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

print("✅ Secrets retrieved successfully from Key Vault")
print(f"   Client ID starts with: {client_id[:8]}...")
print(f"   Tenant ID starts with: {tenant_id[:8]}...")
print(f"   Secret retrieved: {'*' * 20} (hidden for security)")

In [0]:
# ── CELL 3: Load Config + Configure Storage Connection ────────
import json

# ─── Step 1: Load config from repo ───────────────────────────
# This reads our config.json so nothing is hardcoded
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ─── Step 2: Read values from config ─────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
silver_path     = config['storage']['silver_path']
gold_path       = config['storage']['gold_path']
scope_name      = config['secret_scope']

print("✅ Config loaded successfully")
print(f"   Environment:     {config['environment']}")
print(f"   Storage Account: {storage_account}")
print(f"   Bronze Path:     {bronze_path}")
print(f"   Silver Path:     {silver_path}")
print(f"   Gold Path:       {gold_path}")

# ─── Step 3: Retrieve secrets dynamically ────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

print("✅ Secrets retrieved from Key Vault")

# ─── Step 4: Configure Spark to connect to ADLS ──────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("✅ Spark configured for ADLS connection")
print(f"   Auth Type: OAuth (Service Principal)")

In [0]:
# ── CELL 4: Test Storage Container Access ─────────────────────
# Verify Spark can actually reach bronze/silver/gold containers

containers = {
    "Bronze": bronze_path,
    "Silver": silver_path,
    "Gold":   gold_path
}

print("Testing storage container access:")
print("-" * 50)

all_passed = True

for layer, path in containers.items():
    try:
        dbutils.fs.ls(path)
        print(f"  ✅ {layer}: Accessible")
    except Exception as e:
        print(f"  ❌ {layer}: FAILED")
        print(f"     Error: {str(e)}")
        all_passed = False

print("-" * 50)

if all_passed:
    print("✅ All containers accessible!")
    print("   ADLS Gen2 connection working perfectly")
else:
    print("❌ Some containers failed")
    print("   Check SP permissions on storage account")

In [0]:
# ── CELL 5: Write First Delta Table to Bronze ─────────────────
# Simulating what a real pipeline does:
# Land raw data into Bronze layer as Delta format

from pyspark.sql import Row
from datetime import datetime

# Create a simple test DataFrame
# This simulates store data coming from ERP system
test_data = [
    Row(store_id=1, store_name="RetailMax Hyderabad",  
        country="India",     region="South", is_active=True),
    Row(store_id=2, store_name="RetailMax Mumbai",     
        country="India",     region="West",  is_active=True),
    Row(store_id=3, store_name="RetailMax London",     
        country="UK",        region="North", is_active=True),
    Row(store_id=4, store_name="RetailMax New York",   
        country="USA",       region="East",  is_active=True),
    Row(store_id=5, store_name="RetailMax Sydney",     
        country="Australia", region="East",  is_active=False),
]

# Create DataFrame
stores_df = spark.createDataFrame(test_data)

print("DataFrame created:")
print("-" * 50)
stores_df.show()
stores_df.printSchema()

In [0]:
# ── CELL 6: Write DataFrame to Bronze as Delta Table ──────────
# This is what EVERY pipeline does:
# Raw data → Delta format → Bronze container

# Define output path in Bronze layer
output_path = bronze_path + "test/stores"

# Write as Delta format
(stores_df
    .write
    .format("delta")           # Always Delta in our project
    .mode("overwrite")         # Overwrite for this test
    .save(output_path)
)

print("✅ Delta table written successfully!")
print(f"   Path: {output_path}")
print()
print("Why Delta and not Parquet or CSV?")
print("  → ACID transactions guaranteed")
print("  → Transaction log created automatically")
print("  → Can time travel to previous versions")
print("  → Schema enforcement built-in")
print("  → Can UPDATE/DELETE/MERGE records")

In [0]:
# ── CELL 7: Explore Delta Table Structure ─────────────────────
# This is one of the most important things to understand
# What does Delta ACTUALLY create on disk?

output_path = bronze_path + "test/stores"

print("Files created by Delta Lake:")
print("=" * 60)

# List ALL files created
all_files = dbutils.fs.ls(output_path)

for f in all_files:
    print(f"  📁 {f.name}")

print()
print("=" * 60)
print("Now let's look inside _delta_log:")
print("=" * 60)

# Look inside the transaction log
delta_log_files = dbutils.fs.ls(output_path + "/_delta_log")

for f in delta_log_files:
    print(f"  📄 {f.name}  ({f.size} bytes)")

print()
print("What is _delta_log?")
print("  → Every operation recorded as JSON file")
print("  → This is how Delta knows what changed")
print("  → This enables time travel")
print("  → This is what makes Delta ACID compliant")
print("  → Parquet files do NOT have this")

In [0]:
# ── CELL 8: Peek Inside Delta Transaction Log ─────────────────
log_file_path = bronze_path + "test/stores/_delta_log/00000000000000000000.json"

log_content = spark.read.text(log_file_path)

print("Contents of transaction log (first entry):")
print("=" * 60)
log_content.show(truncate=False)

In [0]:
# ── CELL 9: Read Delta Table Back ─────────────────────────────
# Prove we can read what we wrote

read_df = spark.read.format("delta").load(output_path)

print("✅ Delta table read back successfully")
print(f"   Record count: {read_df.count()}")
print("-" * 50)
read_df.show()

In [0]:
# ── CELL 10: Query Delta Table Using SQL ──────────────────────
# Register as temp view so we can use SQL syntax

read_df.createOrReplaceTempView("stores")

result = spark.sql("""
    SELECT 
        store_id,
        store_name,
        country,
        region
    FROM stores
    WHERE is_active = true
    ORDER BY store_id
""")

print("✅ SQL query executed successfully:")
result.show()

In [0]:
# ── CELL 11: Delta Time Travel - View History ─────────────────
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, output_path)

print("Delta Table History:")
print("=" * 60)
delta_table.history().show(truncate=False)

In [0]:
# ── CELL 12: Time Travel - Query Old Version ──────────────────
# Read the table AS IT WAS in version 0

old_version_df = spark.read.format("delta") \
    .option("versionAsOf", 0) \
    .load(output_path)

print("Data as it existed in VERSION 0:")
old_version_df.show()

print()
print("Data as it exists NOW (version 1):")
current_df = spark.read.format("delta").load(output_path)
current_df.show()

In [0]:
# ── CELL 13: Create a REAL Change - Add New Store ─────────────
# Simulate a new store opening (real business scenario)

new_data = [
    Row(store_id=1, store_name="RetailMax Hyderabad",  
        country="India",     region="South", is_active=True),
    Row(store_id=2, store_name="RetailMax Mumbai",     
        country="India",     region="West",  is_active=True),
    Row(store_id=3, store_name="RetailMax London",     
        country="UK",        region="North", is_active=True),
    Row(store_id=4, store_name="RetailMax New York",   
        country="USA",       region="East",  is_active=True),
    Row(store_id=5, store_name="RetailMax Sydney",     
        country="Australia", region="East",  is_active=False),
    Row(store_id=6, store_name="RetailMax Dubai",     
        country="UAE", region="Middle East",  is_active=True),  # NEW STORE
]

updated_df = spark.createDataFrame(new_data)

(updated_df
    .write
    .format("delta")
    .mode("overwrite")
    .save(output_path)
)

print("✅ New store added - Dubai (store_id=6)")
print("   This creates a NEW version of the table")

In [0]:
# ── CELL 14: Check History Again ───────────────────────────────
delta_table = DeltaTable.forPath(spark, output_path)

print("Updated Delta Table History:")
delta_table.history().select(
    "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)

In [0]:
# ── CELL 15: Compare Old vs New Using Time Travel ─────────────

print("VERSION 1 (before Dubai was added):")
print("-" * 50)
old_df = spark.read.format("delta") \
    .option("versionAsOf", 1) \
    .load(output_path)
old_df.orderBy("store_id").show()

print()
print("CURRENT VERSION (after Dubai added):")
print("-" * 50)
current_df = spark.read.format("delta").load(output_path)
current_df.orderBy("store_id").show()

In [0]:
# ── CELL 16: Day 2 Summary ────────────────────────────────────
print("=" * 60)
print("DAY 2 - ENVIRONMENT VALIDATION COMPLETE")
print("=" * 60)
print()
print("✅ Spark 3.5.0 running on Databricks 14.3 LTS")
print("✅ Config loaded from GitHub (config-driven)")
print("✅ Secrets retrieved from Azure Key Vault")
print("✅ Spark configured with Service Principal OAuth")
print("✅ ADLS Gen2 Bronze/Silver/Gold accessible")
print("✅ Delta table written to Bronze container")
print("✅ Delta transaction log explored")
print("✅ Data skipping stats understood")
print("✅ Time travel proved with real data change")
print("✅ SQL queries working on Delta tables")
print()
print("Architecture proven end-to-end:")
print()
print("  GitHub Config")
print("       ↓")
print("  Key Vault Secrets")
print("       ↓")
print("  Service Principal Auth")
print("       ↓")
print("  ADLS Gen2 Storage")
print("       ↓")
print("  Delta Lake Tables")
print("       ↓")
print("  Spark SQL Queries")
print()
print("Ready for Week 2: Bronze Layer Pipelines!")
print("=" * 60)